# Importing libraries

In [1]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from memory_profiler import memory_usage

# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [2]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [3]:
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")

train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45615 entries, 0 to 45614
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    45615 non-null  object
 1   label   45615 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 712.9+ KB


# Dataset preprocessing

In [4]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [5]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultinomialNB(),
        'params': {
            'alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 150, 200],
            'criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': SGDClassifier(),
        'params': {
            'alpha': [0.0001, 0.001, 0.01],
            'penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [6]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = sorted(train['label'].unique())
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [7]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train['label'])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                accuracy = accuracy_score(val['label'], y_pred)
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val['label'], y_pred, average=None, labels=classes, zero_division=0)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multiclass1.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 76.19586860004347
Peak memory usage during training: 562.30078125 MB
Prediction time: 1.133649300027173
Peak memory usage during prediction: 556.421875 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_3720\1197191698.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 75.1951239000191
Peak memory usage during training: 567.50390625 MB
Prediction time: 1.0976236999849789
Peak memory usage during prediction: 563.453125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 75.31954990001395
Peak memory usage during training: 573.76171875 MB
Prediction time: 1.0989768000436015
Peak memory usage during prediction: 559.41015625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 143.82616779999807
Peak memory usage during training: 731.71484375 MB
Prediction time: 1.3263453000108711
Peak memory usage during prediction: 732.0625 MB
------------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 0.8293976999702863
Peak memory usage during training: 936.87890625 MB
Prediction time: 1.8185971000348218
Peak memory usage during prediction: 2101.359375 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 7.815516300033778
Peak memory usage during training: 201.23828125 MB
Prediction time: 1.0453059999854304
Peak memory usage during prediction: 177.37890625 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 7.796173200011253
Peak memory usage during training: 201.0078125 MB
Prediction time: 1.1603092000004835
Peak memory usage during prediction: 177.21484375 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 7.897672499995679
Peak memory usage during training: 201.18359375 MB
Prediction time: 1.0959071000106633
Peak memory usage during prediction: 177.390625 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'alpha': 0.0001, 'penalty': 'l2'} and vectorizer CountVectorizer
Training time: 1.324422899982892
Peak memory usage during training: 200.4921875 MB
Prediction time: 1.4696710999705829
Peak memory usage during prediction: 157.546875 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'alpha': 0.0001, 'penalty': 'l1'} and vectorizer CountVectorizer
Training time: 1.4402074000099674
Peak memory usage during training: 200.73046875 MB
Prediction time: 1.9452456000144593
Peak memory usage during prediction: 157.546875 MB
---------------------------------------------------------------------------------

# Process results

In [8]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seed                    396 non-null    object 
 1   vectorizer              396 non-null    object 
 2   model                   396 non-null    object 
 3   params                  396 non-null    object 
 4   accuracy                396 non-null    float64
 5   training_time           396 non-null    float64
 6   prediction_time         396 non-null    float64
 7   peak_memory_train       396 non-null    float64
 8   peak_memory_prediction  396 non-null    float64
 9   precision_class_0       396 non-null    float64
 10  recall_class_0          396 non-null    float64
 11  f1_class_0              396 non-null    float64
 12  precision_class_1       396 non-null    float64
 13  recall_class_1          396 non-null    float64
 14  f1_class_1              396 non-null    fl

In [9]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,recall_class_0,f1_class_0,precision_class_1,recall_class_1,f1_class_1,precision_class_2,recall_class_2,f1_class_2
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.6175,76.195869,1.133649,562.300781,556.421875,0.552941,0.150641,0.236776,0.565842,0.815880,0.668238,0.723565,0.584860,0.646860
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.6245,75.195124,1.097624,567.503906,563.453125,0.647059,0.176282,0.277078,0.567546,0.817031,0.669811,0.728916,0.590965,0.652731
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.6245,75.319550,1.098977,573.761719,559.410156,0.647059,0.176282,0.277078,0.567546,0.817031,0.669811,0.728916,0.590965,0.652731
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.6255,143.826168,1.326345,731.714844,732.062500,0.597826,0.176282,0.272277,0.573529,0.807825,0.670807,0.722222,0.603175,0.657352
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.6240,150.828404,1.264758,743.062500,743.003906,0.654762,0.176282,0.277778,0.567891,0.818182,0.670438,0.725904,0.588523,0.650034


In [10]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,recall_class_0,f1_class_0,precision_class_1,recall_class_1,f1_class_1,precision_class_2,recall_class_2,f1_class_2,f1_avg
0,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.01}",3.333333,0.5175,5.199100,1.017611,723.993490,680.812500,0.000000,0.000000,0.000000,0.476434,0.965478,0.638023,0.820084,0.239316,0.370510,0.336178
1,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.1}",3.333333,0.5305,5.316520,1.017921,723.964844,680.927083,0.500000,0.003205,0.006369,0.484229,0.953970,0.642387,0.807692,0.282051,0.418100,0.355619
2,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 1.0}",3.333333,0.5885,5.305759,1.006409,723.990885,680.789062,0.470238,0.253205,0.329167,0.540340,0.840046,0.657658,0.765073,0.449328,0.566154,0.517659
3,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.01}",3.333333,0.5175,7.460948,1.021946,723.815104,688.048177,0.000000,0.000000,0.000000,0.476434,0.965478,0.638023,0.820084,0.239316,0.370510,0.336178
4,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.1}",3.333333,0.5405,7.534950,1.062201,723.789062,688.020833,0.800000,0.025641,0.049689,0.489869,0.945915,0.645465,0.804487,0.306471,0.443855,0.379670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'rbf'}",3.333333,0.6440,459.653254,6.704309,1147.979167,887.363281,0.630252,0.240385,0.348028,0.579688,0.853855,0.690554,0.783694,0.575092,0.663380,0.567321
128,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'sigmoid'}",3.333333,0.6445,343.193950,4.797755,1121.990885,881.872396,0.552486,0.320513,0.405680,0.595960,0.814730,0.688381,0.762282,0.587302,0.663448,0.585836
129,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'linear'}",3.333333,0.5995,1332.701071,4.549585,1163.554688,701.381510,0.477193,0.435897,0.455611,0.573501,0.682394,0.623226,0.690162,0.573871,0.626667,0.568502
130,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'rbf'}",3.333333,0.6465,679.867944,7.032474,937.277344,710.782552,0.575758,0.304487,0.398323,0.595034,0.799770,0.682376,0.754123,0.614164,0.676985,0.585895


In [11]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train['label'])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test['label'], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test['label'], y_pred, average=None, labels=classes, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: SVC
Best model params: {'C': 0.1, 'kernel': 'linear'}
Best vectorizer: CountVectorizer
Best accuracy: 0.5825464018235103

Class 0
Precision: 0.6826983135540288
Recall: 0.27517623363544813
F1: 0.39224834021173516
Support: 3972

Class 1
Precision: 0.5647874516935667
Recall: 0.8369546909213408
F1: 0.6744485917882592
Support: 5937

Class 2
Precision: 0.5803713527851458
Recall: 0.46063157894736845
F1: 0.5136150234741784
Support: 2375



In [12]:
with open('models/best_model_sklearn_multiclass1.pkl', 'wb') as f:
    pickle.dump(pipeline, f)